# Lab: Shodan Excel Analytics

Este notebook transforma resultados do Shodan em uma analise tabular e visual no estilo Excel.

Ele funciona em dois modos:

- **sample**: usa um CSV didatico incluido no repositorio. Nao precisa de chave Shodan.
- **shodan**: usa a API real do Shodan quando `SHODAN_API_KEY` estiver configurada.

Objetivo: aprender a transformar resultados brutos em dataset, graficos, score didatico, resumo executivo e arquivos exportaveis.

> Use este lab apenas para analise defensiva e educacional. Nao use consultas ofensivas, dados reais sensiveis ou chaves expostas em celulas versionadas.


## 1. Configuracao

No Colab, use `Runtime > Run all` ou execute celula por celula.

Para usar Shodan real, configure `SHODAN_API_KEY` como secret do Colab ou variavel de ambiente. Sem chave, o notebook usa automaticamente os dados de exemplo.


In [ ]:
# Dependencias usadas pelo lab. No Colab normalmente ja existem, mas esta celula deixa o ambiente reprodutivel.
%pip install -q pandas matplotlib requests openpyxl


In [ ]:
from __future__ import annotations

import json
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
plt.style.use("default")

RAW_SAMPLE_URL = "https://raw.githubusercontent.com/rodrigorosamuniz/labs-seguranca-notebooks/main/labs/shodan-excel-analytics/data/sample_shodan_results.csv"
LOCAL_SAMPLE_PATH = Path("data/sample_shodan_results.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## 2. Parametros Do Lab

Ajuste os parametros abaixo antes de executar.

- `MODE = "sample"`: usa dados de exemplo.
- `MODE = "shodan"`: tenta consultar a API real.
- `MODE = "auto"`: usa Shodan se houver chave; caso contrario, usa exemplo.

Para controlar custo, mantenha `MAX_RESULTS` baixo. A API de busca do Shodan consome creditos conforme volume de resultados baixados.


In [ ]:
MODE = "auto"  # sample, shodan ou auto
SHODAN_QUERY = 'apache country:BR'
COMPARE_QUERIES = ['apache', 'nginx', 'port:3389', 'product:OpenSSH']
MAX_RESULTS = 100
TOP_N = 10
EXPORT_XLSX = True
EXPORT_MARKDOWN_REPORT = True


## 3. Funcoes De Coleta

Esta secao carrega dados de exemplo ou consulta o Shodan real.

O notebook nao salva sua API key. Nunca cole sua chave diretamente em uma celula versionada.


In [ ]:
def get_shodan_api_key() -> str | None:
    key = os.getenv("SHODAN_API_KEY")
    if key:
        return key

    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("SHODAN_API_KEY")
        if key:
            return key
    except Exception:
        pass

    return None


def fetch_shodan_api_info(api_key: str) -> dict[str, Any]:
    response = requests.get(
        "https://api.shodan.io/api-info",
        params={"key": api_key},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def estimate_query_credits(max_results: int) -> int:
    # Shodan normalmente pagina resultados em blocos de ate 100 itens para busca.
    return max(1, (max_results + 99) // 100)


def fetch_shodan_search(api_key: str, query: str, max_results: int = 100) -> list[dict[str, Any]]:
    results: list[dict[str, Any]] = []
    page = 1

    while len(results) < max_results:
        response = requests.get(
            "https://api.shodan.io/shodan/host/search",
            params={"key": api_key, "query": query, "page": page},
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        matches = payload.get("matches", [])
        if not matches:
            break

        for item in matches:
            item["query_label"] = query
            results.append(item)
            if len(results) >= max_results:
                break
        page += 1

    return results


def load_sample_data() -> pd.DataFrame:
    if LOCAL_SAMPLE_PATH.exists():
        return pd.read_csv(LOCAL_SAMPLE_PATH)
    return pd.read_csv(RAW_SAMPLE_URL)


## 4. Normalizacao

Resultados do Shodan sao ricos, mas variam por servico. Esta funcao cria uma tabela estavel para analise no estilo Excel.


In [ ]:
def normalize_shodan_matches(matches: list[dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for item in matches:
        location = item.get("location") or {}
        vulns = item.get("vulns") or {}
        hostnames = item.get("hostnames") or []
        opts = item.get("opts") or {}

        rows.append({
            "query_label": item.get("query_label", "shodan"),
            "ip": item.get("ip_str") or item.get("ip"),
            "port": item.get("port"),
            "transport": item.get("transport"),
            "org": item.get("org") or "Unknown",
            "asn": item.get("asn") or "Unknown",
            "country": location.get("country_code") or location.get("country_name") or "Unknown",
            "city": location.get("city") or "Unknown",
            "product": item.get("product") or item.get("_shodan", {}).get("module") or "Unknown",
            "version": item.get("version") or "",
            "hostnames": ";".join(hostnames) if isinstance(hostnames, list) else str(hostnames),
            "vulns": ";".join(vulns.keys()) if isinstance(vulns, dict) else str(vulns or ""),
            "timestamp": item.get("timestamp") or opts.get("raw", ""),
            "cloud": bool(item.get("cloud") or item.get("tags") and "cloud" in item.get("tags", [])),
        })

    return pd.DataFrame(rows)


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.copy()
    expected = ["query_label", "ip", "port", "transport", "org", "asn", "country", "city", "product", "version", "hostnames", "vulns", "timestamp", "cloud"]
    for column in expected:
        if column not in normalized.columns:
            normalized[column] = ""

    normalized["port"] = pd.to_numeric(normalized["port"], errors="coerce").fillna(0).astype(int)
    normalized["cloud"] = normalized["cloud"].astype(str).str.lower().isin(["true", "1", "yes"])
    normalized["vuln_count"] = normalized["vulns"].fillna("").astype(str).apply(lambda value: 0 if value.strip() == "" else len([v for v in value.split(";") if v.strip()]))
    normalized["has_vulns"] = normalized["vuln_count"] > 0
    normalized["has_version"] = normalized["version"].fillna("").astype(str).str.strip() != ""
    normalized["product"] = normalized["product"].fillna("Unknown").replace("", "Unknown")
    normalized["org"] = normalized["org"].fillna("Unknown").replace("", "Unknown")
    normalized["country"] = normalized["country"].fillna("Unknown").replace("", "Unknown")
    return normalized[expected + ["vuln_count", "has_vulns", "has_version"]]


## 5. Coleta Com Controle De Creditos

Esta celula decide o modo de execucao, mostra informacoes da conta quando possivel e limita a busca real.


In [ ]:
api_key = get_shodan_api_key()
mode = MODE.lower().strip()

if mode == "auto":
    mode = "shodan" if api_key else "sample"

print(f"Modo selecionado: {mode}")
print(f"MAX_RESULTS: {MAX_RESULTS}")
print(f"Estimativa didatica de creditos de busca: {estimate_query_credits(MAX_RESULTS)}")

if mode == "shodan":
    if not api_key:
        raise RuntimeError("SHODAN_API_KEY nao configurada. Use MODE='sample' ou configure a chave.")

    info = fetch_shodan_api_info(api_key)
    print("Informacoes da conta Shodan:")
    display(pd.DataFrame([info]))

    if COMPARE_QUERIES:
        all_matches = []
        per_query_limit = max(1, MAX_RESULTS // len(COMPARE_QUERIES))
        for query in COMPARE_QUERIES:
            print(f"Consultando: {query} | limite: {per_query_limit}")
            all_matches.extend(fetch_shodan_search(api_key, query, per_query_limit))
        df = prepare_dataframe(normalize_shodan_matches(all_matches))
    else:
        matches = fetch_shodan_search(api_key, SHODAN_QUERY, MAX_RESULTS)
        df = prepare_dataframe(normalize_shodan_matches(matches))
else:
    df = prepare_dataframe(load_sample_data())

print(f"Registros carregados: {len(df)}")
display(df.head(10))


## 6. Analise Estilo Excel

As proximas celulas criam tabelas resumidas equivalentes a tabelas dinamicas simples.


In [ ]:
def count_by(df: pd.DataFrame, column: str, top_n: int = 10) -> pd.DataFrame:
    return (
        df.groupby(column, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(top_n)
    )

by_country = count_by(df, "country", TOP_N)
by_port = count_by(df, "port", TOP_N)
by_org = count_by(df, "org", TOP_N)
by_product = count_by(df, "product", TOP_N)
by_asn = count_by(df, "asn", TOP_N)
by_query = count_by(df, "query_label", TOP_N)

display(by_country)
display(by_port)
display(by_org)
display(by_product)


In [ ]:
def plot_bar(table: pd.DataFrame, label_column: str, title: str, color: str = "#2F75B5") -> None:
    if table.empty:
        print(f"Sem dados para {title}")
        return

    ax = table.sort_values("count").plot.barh(
        x=label_column,
        y="count",
        legend=False,
        figsize=(9, 4),
        color=color,
    )
    ax.set_title(title)
    ax.set_xlabel("Quantidade")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

plot_bar(by_country, "country", f"Top {TOP_N} paises")
plot_bar(by_port, "port", f"Top {TOP_N} portas", "#70AD47")
plot_bar(by_org, "org", f"Top {TOP_N} organizacoes", "#FFC000")
plot_bar(by_product, "product", f"Top {TOP_N} produtos/servicos", "#C55A11")
plot_bar(by_asn, "asn", f"Top {TOP_N} ASNs", "#8064A2")


## 7. Score Didatico De Risco

Este score nao substitui analise profissional. Ele serve para ensinar priorizacao.


In [ ]:
SENSITIVE_PORTS = {21, 22, 23, 25, 110, 143, 445, 1433, 1521, 2375, 3306, 3389, 5432, 5900, 6379, 9200, 27017}
REMOTE_ACCESS_PORTS = {22, 23, 3389, 5900}
DATABASE_PORTS = {1433, 1521, 3306, 5432, 6379, 9200, 27017}

def risk_reasons(row: pd.Series) -> list[str]:
    reasons = []
    port = int(row.get("port", 0))

    if port in SENSITIVE_PORTS:
        reasons.append("porta sensivel")
    if port in REMOTE_ACCESS_PORTS:
        reasons.append("acesso remoto")
    if port in DATABASE_PORTS:
        reasons.append("servico de dados")
    if row.get("has_vulns"):
        reasons.append("CVE informada")
    if not row.get("has_version"):
        reasons.append("produto sem versao")
    if row.get("cloud"):
        reasons.append("exposicao em cloud")

    return reasons

def risk_score(row: pd.Series) -> int:
    score = 0
    port = int(row.get("port", 0))

    if port in SENSITIVE_PORTS:
        score += 30
    if port in REMOTE_ACCESS_PORTS:
        score += 20
    if port in DATABASE_PORTS:
        score += 25
    if row.get("has_vulns"):
        score += 40
    if not row.get("has_version"):
        score += 10
    if row.get("cloud"):
        score += 10

    return min(score, 100)

df["risk_score"] = df.apply(risk_score, axis=1)
df["risk_reasons"] = df.apply(lambda row: "; ".join(risk_reasons(row)), axis=1)

risk_view = df.sort_values(["risk_score", "vuln_count"], ascending=False)[["ip", "port", "org", "country", "product", "version", "vulns", "risk_score", "risk_reasons"]]
display(risk_view.head(15))
plot_bar(count_by(df, "risk_score", TOP_N), "risk_score", "Distribuicao por score de risco", "#C00000")


## 8. Comparacao Entre Queries

Quando o dataset possui `query_label`, o notebook compara perfis de exposicao por consulta.


In [ ]:
query_port_matrix = pd.crosstab(df["query_label"], df["port"])
query_product_matrix = pd.crosstab(df["query_label"], df["product"])

print("Portas por query")
display(query_port_matrix)

print("Produtos por query")
display(query_product_matrix)

if not by_query.empty:
    plot_bar(by_query, "query_label", "Resultados por query", "#4472C4")


## 9. Analise De CVEs

Se o Shodan ou o CSV trouxerem CVEs, esta secao resume ocorrencias e hosts afetados.


In [ ]:
def explode_vulnerabilities(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        vulns = [v.strip() for v in str(row.get("vulns", "")).split(";") if v.strip()]
        for vuln in vulns:
            rows.append({
                "cve": vuln,
                "ip": row.get("ip"),
                "port": row.get("port"),
                "org": row.get("org"),
                "product": row.get("product"),
                "risk_score": row.get("risk_score"),
            })
    return pd.DataFrame(rows)

vuln_df = explode_vulnerabilities(df)

if vuln_df.empty:
    print("Nenhuma CVE informada no dataset.")
else:
    by_cve = count_by(vuln_df, "cve", TOP_N)
    display(by_cve)
    display(vuln_df.sort_values(["risk_score", "cve"], ascending=[False, True]).head(20))
    plot_bar(by_cve, "cve", f"Top {TOP_N} CVEs", "#A5A5A5")


## 10. Resumo Executivo Automatico

Esta celula gera um texto curto que pode ser usado como base de relatorio.


In [ ]:
def first_value(table: pd.DataFrame, column: str) -> str:
    if table.empty:
        return "sem dados"
    return str(table.iloc[0][column])

summary_lines = [
    f"A analise carregou {len(df)} registros no modo {mode}.",
    f"A porta mais recorrente foi {first_value(by_port, 'port')}.",
    f"O pais mais recorrente foi {first_value(by_country, 'country')}.",
    f"A organizacao mais recorrente foi {first_value(by_org, 'org')}.",
    f"O produto/servico mais recorrente foi {first_value(by_product, 'product')}.",
    f"Hosts com CVE informada: {int(df['has_vulns'].sum())}.",
    f"Maior score didatico observado: {int(df['risk_score'].max()) if not df.empty else 0}.",
]

executive_summary = "\n".join(f"- {line}" for line in summary_lines)
print(executive_summary)


## 11. Export Excel E Relatorio Markdown

A exportacao gera artefatos para entrega de exercicio ou relatorio.


In [ ]:
def export_excel(path: Path) -> None:
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="raw_results", index=False)
        by_country.to_excel(writer, sheet_name="by_country", index=False)
        by_port.to_excel(writer, sheet_name="by_port", index=False)
        by_org.to_excel(writer, sheet_name="by_org", index=False)
        by_product.to_excel(writer, sheet_name="by_product", index=False)
        by_asn.to_excel(writer, sheet_name="by_asn", index=False)
        risk_view.to_excel(writer, sheet_name="risk_scores", index=False)
        if not vuln_df.empty:
            vuln_df.to_excel(writer, sheet_name="vulnerabilities", index=False)
        pd.DataFrame({"summary": summary_lines}).to_excel(writer, sheet_name="summary", index=False)

def export_markdown_report(path: Path) -> None:
    content = [
        "# Shodan Excel Analytics - Resumo\n",
        executive_summary,
        "\n## Top Portas\n",
        by_port.to_markdown(index=False),
        "\n## Top Produtos\n",
        by_product.to_markdown(index=False),
        "\n## Maiores Scores\n",
        risk_view.head(10).to_markdown(index=False),
    ]
    path.write_text("\n".join(content), encoding="utf-8")

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

if EXPORT_XLSX:
    xlsx_path = OUTPUT_DIR / f"shodan_excel_analytics_{timestamp}.xlsx"
    export_excel(xlsx_path)
    print(f"Excel exportado: {xlsx_path}")

if EXPORT_MARKDOWN_REPORT:
    report_path = OUTPUT_DIR / f"shodan_summary_{timestamp}.md"
    export_markdown_report(report_path)
    print(f"Relatorio Markdown exportado: {report_path}")


## 12. Perguntas Guiadas

Responda individualmente depois de rodar a analise:

1. Qual porta aparece com maior frequencia? Isso indica risco automaticamente?
2. Qual organizacao concentra mais exposicoes?
3. Quais produtos aparecem sem versao? Por que isso dificulta priorizacao?
4. Quantos hosts aparecem com CVE informada?
5. Qual host recebeu maior score didatico? O motivo do score faz sentido?
6. Que grafico voce usaria para explicar o resultado para uma pessoa nao tecnica?
7. O que voce investigaria antes de afirmar que existe uma vulnerabilidade exploravel?
8. Que limitacoes existem em dados obtidos pelo Shodan?
